In [27]:
import requests
from requests.adapters import HTTPAdapter
from requests.packages.urllib3.util.retry import Retry

import pandas as pd
import json
import csv
import pickle
import numpy as np

counter = 0
core_name = "dc_cubes_historic"
predictionColumn = "cpuusage_ps"


In [127]:
def pushData(row):
    global core_name
    global counter
    global allMetrics
    global predictionColumn
    # defining the api-endpoint
    url = "http://localhost:8983/solr/"+core_name+"/update/json/docs"
    # data to be sent to api
    data = {
                "timestamp": str(row["timestamp"]),
                "cluster": row["cluster"],
                "dc": row["dc"],
                "perm": -1, #row["perm"],
                "instanz": row["instanz"],
                "verfahren": "-1",# row["verfahren"],
                "service": "-1" ,#row["verfahren"],
                "response": -1 ,#row["response"],
                "count": row[predictionColumn],
                "minv":  -1,#row["minv"],
                "maxv":  -1,#row["maxv"],
                "avg": -1, #row["avg"],
                "var": -1 ,#row["var"],
                "dev_upp": -1, #row["dev_upp"],
                "dev_low": -1, #row["dev_low"],
                "perc90": -1 ,#row["perc90"],
                "perc95": -1 ,#row["perc95"],
                "perc99": -1 ,#row["perc99.9"],
                "sum": -1, #row["sum"],
                "sum_of_squares": -1, #row["sum_of_squares"],
                "server": "PBZ0%dE00_PERM02_S0%s_OSB" % (row["cluster"], row["instanz"]), #row["server"]
            }
    
    for metric in allMetrics:
        data[metric] = row[metric]
    session = requests.Session()
    retry = Retry(connect=3, backoff_factor=0.5)
    adapter = HTTPAdapter(max_retries=retry)
    session.mount('http://', adapter)
    session.mount('https://', adapter)

    session.get(url)
    headers = {'Content-type': 'application/json'}
    # sending post request
    session.post(url=url, data=json.dumps(data), headers=headers)
    counter += 1
    if (counter % 1000 == 0):
        print("Commiting... counter:", counter)
        requests.get("http://localhost:8983/solr/"+core_name+"/update?commit=true")

In [29]:
def createSolrCore(core_name):
    url = "http://localhost:8983/solr/admin/cores?action=CREATE&name=" + \
        core_name+"&configSet=_default"
    requests.post(url=url)
    print(core_name, " created")



In [30]:
def initSchema(core_name, allMetrics):
    url = "http://localhost:8983/solr/"+core_name+"/schema"
    headers = {'Content-type': 'application/json'}
    rowsDict = {
        "timestamp": "pdate", "host": "string", "cluster": "pint", "dc": "pint", "perm": "pint", "instanz": "string", "verfahren": "string",
        "service": "string", "response": "pint", "count": "pint", "minv": "pint", "maxv": "pint", "avg": "pfloat", "var": "pfloat",
        "dev_upp": "pfloat", "dev_low": "pfloat", "perc90": "pfloat", "perc95": "pfloat", "perc99": "pfloat", "sum": "pint",
        "sum_of_squares": "pint", "server": "string"}

    for name in rowsDict:
        data = {
            "add-field": {"stored": "true", "docValues": "true", "indexed": "false", "multiValued": "false", "name": name, "type": rowsDict[name]}
        }
        requests.post(url=url, data=json.dumps(data), headers=headers)
        
    for metric in allMetrics:
        data = {
            "add-field": {"stored": "true", "docValues": "true", "indexed": "false", "multiValued": "false", "name": metric, "type": "pfloat"}
        }
        requests.post(url=url, data=json.dumps(data), headers=headers)
    
    
    print(core_name, " schema inited")


In [61]:
def deleteCoreDocuments(core_name):
    url = "http://localhost:8983/solr/"+core_name + \
        "/update?commitWithin=1000&overwrite=true&wt=json"
    headers = {'Content-type': 'application/json'}
    data = {'delete': {'query': '*:*'}}
    requests.post(url=url, data=json.dumps(data), headers=headers)
    print("deleted old documents from "+core_name+" core")

In [6]:
# get existing solr cores
url = "http://localhost:8983/solr/admin/cores?action=STATUS"
response = requests.get(url).json()
activeCores = response['status'].keys()

In [118]:
if core_name in activeCores:
    print(core_name + " already exists")
    # delete old data/predictions
    #deleteCoreDocuments(core_name)
# else forecast core doesn't exist
else:
    print(core_name + " doesn't exist")
    # create an new forecast solr core
    createSolrCore(core_name)
    # init schema
    initSchema(core_name)

dc_cubes_historic already exists
deleted old documents from dc_cubes_historic core


In [119]:
# Read data from pickle file
with open("./4week_transformed_droppedErrors_filled.pkl", "rb") as pickleFile:
    df = pickle.load(pickleFile)
df = df.reset_index()

In [120]:
allMetrics = df.columns

In [121]:
allMetrics = allMetrics.drop("timestamp")

In [122]:
initSchema(core_name, allMetrics)

dc_cubes_historic  schema inited


In [123]:
# There are two dc's: 0 and 1
# dc 0 has clusters 6 and 8
# dc 1 has clusters 5 and 7
# Every Cluster has all 8 instances

def pushDataForAllInstances():
    instances = ["1","2","3","4","5","6","7","8" ]
    for instanz in instances:
        df["instanz"] = instanz
        df.apply(pushData, axis=1)

In [124]:
df["dc"] = 0

In [125]:
df["cluster"] = 6

In [128]:
pushDataForAllInstances()

Commiting... counter: 93000
Commiting... counter: 94000
Commiting... counter: 95000
Commiting... counter: 96000
Commiting... counter: 97000
Commiting... counter: 98000
Commiting... counter: 99000
Commiting... counter: 100000
Commiting... counter: 101000
Commiting... counter: 102000
Commiting... counter: 103000
Commiting... counter: 104000
Commiting... counter: 105000
Commiting... counter: 106000
Commiting... counter: 107000
Commiting... counter: 108000
Commiting... counter: 109000
Commiting... counter: 110000
Commiting... counter: 111000
Commiting... counter: 112000
Commiting... counter: 113000
Commiting... counter: 114000


In [129]:
df["cluster"] = 8

In [130]:
pushDataForAllInstances()

Commiting... counter: 115000
Commiting... counter: 116000
Commiting... counter: 117000
Commiting... counter: 118000
Commiting... counter: 119000
Commiting... counter: 120000
Commiting... counter: 121000
Commiting... counter: 122000
Commiting... counter: 123000
Commiting... counter: 124000
Commiting... counter: 125000
Commiting... counter: 126000
Commiting... counter: 127000
Commiting... counter: 128000
Commiting... counter: 129000
Commiting... counter: 130000
Commiting... counter: 131000
Commiting... counter: 132000
Commiting... counter: 133000
Commiting... counter: 134000
Commiting... counter: 135000
Commiting... counter: 136000


In [131]:
df["dc"] = 1

In [132]:
df["cluster"] = 5

In [133]:
pushDataForAllInstances()

Commiting... counter: 137000
Commiting... counter: 138000
Commiting... counter: 139000
Commiting... counter: 140000
Commiting... counter: 141000
Commiting... counter: 142000
Commiting... counter: 143000
Commiting... counter: 144000
Commiting... counter: 145000
Commiting... counter: 146000
Commiting... counter: 147000
Commiting... counter: 148000
Commiting... counter: 149000
Commiting... counter: 150000
Commiting... counter: 151000
Commiting... counter: 152000
Commiting... counter: 153000
Commiting... counter: 154000
Commiting... counter: 155000
Commiting... counter: 156000
Commiting... counter: 157000
Commiting... counter: 158000


In [134]:
df["cluster"] = 7

In [135]:
pushDataForAllInstances()

Commiting... counter: 159000
Commiting... counter: 160000
Commiting... counter: 161000
Commiting... counter: 162000
Commiting... counter: 163000
Commiting... counter: 164000
Commiting... counter: 165000
Commiting... counter: 166000
Commiting... counter: 167000
Commiting... counter: 168000
Commiting... counter: 169000
Commiting... counter: 170000
Commiting... counter: 171000
Commiting... counter: 172000
Commiting... counter: 173000
Commiting... counter: 174000
Commiting... counter: 175000
Commiting... counter: 176000
Commiting... counter: 177000
Commiting... counter: 178000
Commiting... counter: 179000
Commiting... counter: 180000
Commiting... counter: 181000


In [136]:
requests.get("http://localhost:8983/solr/"+core_name+"/update?commit=true")

<Response [200]>

In [137]:
#*______________________________________*#

In [138]:
forecast_core_name = "dc_cubes_forecast"

In [139]:
url = "http://localhost:8983/solr/admin/cores?action=STATUS"
response = requests.get(url).json()
activeCores = response['status'].keys()

# if forecast core exists
if forecast_core_name in activeCores:
    print(forecast_core_name + " already exists")
    # delete old data/predictions
    deleteCoreDocuments(forecast_core_name)
# else forecast core doesn't exist
else:
    print(forecast_core_name + " doesn't exist")
    # create an new forecast solr core
    createSolrCore(forecast_core_name)
    # init schema
    initSchema(forecast_core_name)

dc_cubes_forecast already exists
deleted old documents from dc_cubes_forecast core


In [140]:
def getData(core_name):
    # from 1 Month, there are about 90k entries. 2 clusters * 2 dcs * 8 instances * (4 weeks * 7 days * 24 hours * 4 (15min intervall))
    url = 'http://localhost:8983/solr/'+core_name+'/select?q=*:*&sort=timestamp%20asc&rows=100000'
    response = requests.get(url).json()['response']
    response
    return response['docs']

In [141]:
historicCoreName = "dc_cubes_historic"

In [142]:
hist_df = pd.DataFrame.from_dict(getData(historicCoreName))

In [143]:
# splits the data of each cube from the whole df in its own dataframe
def splitInCubesFrames(df):
    unique_server_names = df.server.unique()
    splitted_frames = []
    for name in unique_server_names:
        new_df = df[df['server'] == name][-history_steps:].copy()
        splitted_frames.append(new_df)
    return splitted_frames


In [147]:
measureInterval = 15 #min
daysToPredict = 5
pred_horizon = (60//measureInterval) * 24 * daysToPredict #5 days (4*24*5), timestep = 15min
hours_history = 8
n_history = (60//measureInterval)*hours_history 
history_steps = n_history
forecast_steps = pred_horizon

In [144]:
hist_df.server

0        PBZ06E00_PERM02_S06_OSB
1        PBZ06E00_PERM02_S07_OSB
2        PBZ06E00_PERM02_S08_OSB
3        PBZ06E00_PERM02_S05_OSB
4        PBZ08E00_PERM02_S03_OSB
                  ...           
89243    PBZ07E00_PERM02_S05_OSB
89244    PBZ07E00_PERM02_S06_OSB
89245    PBZ07E00_PERM02_S07_OSB
89246    PBZ07E00_PERM02_S04_OSB
89247    PBZ07E00_PERM02_S08_OSB
Name: server, Length: 89248, dtype: object

In [ ]:
last_timestamp = hist_df.timestamp.max()
hist_df = hist_df.set_index('timestamp')
hist_df.index = pd.to_datetime(hist_df.index).sort_values()

In [151]:
from keras.models import load_model

# split cubes in own frames
cubes_frames = splitInCubesFrames(hist_df)

# load the trained model
model = load_model('cnn_multistep_multivariate.h5')

In [154]:
def makePredictionFrame(model, cubes_frames, last_timestamp):
    prediction_frames = []
    for cube in cubes_frames:
        # transform pre prediction input

        # extracting information from the current server
        last_timestamp = cube.index[-1]
        server_name = cube['server'].iloc[0]
        cluster = cube['cluster'].iloc[0]
        dc = cube['dc'].iloc[0]
        perm = cube['perm'].iloc[0]
        instanz = cube['instanz'].iloc[0]
        verfahren = cube['verfahren'].iloc[0]
        service = cube['service'].iloc[0]

        dropCols = ["cluster",
                "dc",
                "perm",
                "instanz",
                "verfahren",
                "service",
                "response",
                "minv",
                "maxv", 
                "avg",
                "var",
                "dev_upp",
                "dev_low",
                "perc90",
                "perc95",
                "perc99",
                "sum",
                "sum_of_squares",
                "server"]
        # Converting the index as date
        cube.index = pd.to_datetime(cube.index).sort_values()

        dataset = cube.drop(dropCols,axis=1)
        print(dataset.columns.tolist())
        y = dataset[predictionColumn].copy()
        x = dataset.drop(columns=predictionColumn)
        
        # Standardise
        import numpy as np
        from sklearn.preprocessing import StandardScaler

        dataset = cube.values

        # standardise
        scalerX = StandardScaler()
        scalerX.fit(x)
        x = scalerX.transform(x)
        scalerY = StandardScaler()
       # .reshape(-1, 1) # needed for standardScaler
        scalerY.fit(y.values.reshape(-1,1))
        y = scalerY.transform(y.values.reshape(-1,1))
        
        #PCA
        pcaTransformer = PCA(0.95) # keep 95% variance
        pcaTransformer.fit(x)
        x = pcaTransformer.transform(x)
        
        transformed_df = pd.DataFrame().from_records(x)
        transformed_df[predictionColumn] = y

        # predict
        pred_input = transformed_df
        pred_input = pred_input.reshape(
            (1, pred_input.shape[0], pred_input.shape[1]))
        prediction = model.predict(pred_input)
        pred_input = pred_input.reshape((1, pred_input.shape[0],pred_input.shape[1]))

        prediction = model.predict(pred_input)
        prediction = prediction.reshape((pred_horizon,1))
        prediction = np.hstack((prediction, np.zeros((prediction.shape[0], numberOfFeatures-1), dtype=prediction.dtype)))
        
        print(prediction)
        prediction = scalerY.inverse_transform(prediction)
        print(prediction)
        # transform pre solr
#         prediction = prediction.reshape((forecast_steps, 1))
#         prediction = np.hstack((prediction, np.zeros(
#             (prediction.shape[0], 3), dtype=prediction.dtype)))
#         prediction = prediction = scaler.inverse_transform(prediction)
#         prediction = prediction[:, [0]]
#         int_prediction = prediction.astype(int, copy=True)
#         prediction = int_prediction
#         prediction *= (prediction > 0)

        # make the dataframe
        next_timestamps = pd.date_range(
            start=last_timestamp, periods=forecast_steps+1, freq='15min',  closed='right')
        # create the prediction dataframe for the current server
        d = {'timestamp': next_timestamps, 'cluster': cluster, 'dc': dc,
             'perm': perm, 'instanz': instanz,  'verfahren': verfahren, 'service': service, 'response': 200}
        pred_df = pd.DataFrame(data=d)
        pred_df['count'] = prediction
        pred_df['minv'] = 0
        pred_df['maxv'] = 0
        pred_df['avg'] = 0
        pred_df['var'] = 0
        pred_df['dev_upp'] = 0
        pred_df['dev_low'] = 0
        pred_df['perc90'] = 0
        pred_df['perc95'] = 0
        pred_df['perc99.9'] = 0
        pred_df['sum'] = 0
        pred_df['sum_of_squares'] = 0
        pred_df['server'] = server_name

        pred_df['timestamp'] = pred_df['timestamp'].dt.strftime(
            '%Y-%m-%dT%H:%M:00Z')
        prediction_frames.append(pred_df)
    print("Made predictions")
    return pd.concat(prediction_frames, ignore_index=True)



In [156]:
a = ['count', 'transactions_ps', 'opt_sga_size', 'sga_total', 'fixed_sga', 'java_pool', 'shared_pool', 'buffer_cache', 'pga_total', 'other_sga_memory', 'log_buffer', 'large_pool', 'streams_pool', 'total_pga_allocated', 'hardParseBindMismElTimePs', 'javaExecutionElapsedTimePs', 'plsqlEexecutionElapsedTimePs', 'connManageCallElapsedTimePs', 'failedParseElapsedTimePs', 'dbTimePs', 'hardParseSharCritElTimePs', 'plsqlCompilationElapsedTimePs', 'bgCpuTimePs', 'sqlExecuteElapsedTimePs', 'parseTimeElapsedPs', 'failedParseSharMemElTimePs', 'repeatedBindElapsedTimePs', 'dbCpuPs', 'rmanCpuTimeBackupRestorePs', 'inboundPlsqlRpcElapsedTimePs', 'sequenceLoadElapsedTimePs', 'hardParseElapsedTimePs', 'averageWaitTime', 'totalWaitsFG', 'averageWaitTimeFG', 'totalWaitTimeFG', 'totalWaits', 'totalWaitTime', 'shared_free_pct', 'large_free_pct', 'java_free_pct', 'session_usage', 'logons', 'user_limit', 'opencursors', 'process_usage', 'total_memory', 'genericErrors', 'archiveHungErrors', 'blockCorruptErrors', 'mediaFailureErrors', 'sessTerminateErrors', 'user_cpu_time_cnt', 'avg_user_cpu_time_pct', 'user_wait_time_pct', 'host_cpu_usage_pct', 'other_wait_cnt', 'userio_wait_cnt', 'avg_sync_singleblk_read_latency', 'sortsdisk_ps', 'physreads_pt', 'iombs_ps', 'recurscalls_pt', 'physreadslob_pt', 'logreads_pt', 'sortsdisk_pt', 'consistentreadgets_ps', 'redowrites_ps', 'recurscalls_ps', 'softparse_pct', 'failedparses_ps', 'userrollbackundorec_pt', 'physreadsdir_ps', 'sortsrows_ps', 'dbblkgets_ps', 'redosize_ps', 'dbtime_ps', 'logreads_ps', 'tabscanslong_ps', 'dbtime_pt', 'dbblkgets_pt', 'iorequests_ps', 'redosize_pt', 'executeswoparse_pct', 'networkbytes_ps', 'tabscanslong_pt', 'usercall_pct', 'rows_psort', 'executions_ps', 'usercalls_ps', 'physwriteslob_ps', 'logons_pt', 'opncurs_ps', 'physwritesdir_pt', 'avg_active_sessions', 'parses_ps', 'crblks_ps', 'dbblkchanges_ps', 'enqtimeouts_ps', 'leafnodesplits_pt', 'commits_ps', 'physwrites_pt', 'dbblkchanges_pt', 'branchnodesplits_pt', 'indxscanstotal_ps', 'parses_pt', 'physwrites_ps', 'sortsrows_pt', 'leafnodesplits_ps', 'tabscanstotal_pt', 'tabscanstotal_ps', 'sortsmemory_ps', 'enqreqs_pt', 'hardparses_ps', 'sortsmemory_pt', 'enqwaits_pt', 'crundorecs_ps', 'enqtimeouts_pt', 'physreadsdir_pt', 'crblks_pt', 'executions_pt', 'branchnodesplits_ps', 'enqreqs_ps', 'physreadslob_ps', 'physwriteslob_pt', 'consistentreadchanges_pt', 'commits_pt', 'userrollbackundorec_ps', 'failedparses_pt', 'crundorecs_pt', 'indxscansfull_ps', 'physreads_ps', 'indxscansfull_pt', 'consistentreadchanges_ps', 'hardparses_pt', 'bgcheckpoints_ps', 'logons_ps', 'dbwrcheckpoints_ps', 'usercalls_pt', 'consistentreadgets_pt', 'redowrites_pt', 'enqwaits_ps', 'indxscanstotal_pt', 'enqdeadlocks_pt', 'rollbacks_ps', 'physwritesdir_ps', 'rollbacks_pt', 'opncurs_pt', 'enqdeadlocks_ps', 'avg_users_waiting_on_class', 'dbtime_waitclass_pct', 'interconnect_rate', 'corrupt', 'currentgets_cs', 'lost', 'cr_request_cs', 'time_cs', 'libcache_miss_pct', 'dictionaryhit_pct', 'pxdwngrd75_ps', 'cursorcachehit_pct', 'redologalloc_hit_pct', 'cpuusage_pt', 'pxdwngrd25_ps', 'dictionarymiss_pct', 'response_time_pt', 'bufcachehit_pct', 'cpu_time_pct', 'pxdwngrdserial_ps', 'pgacachehit_pct', 'cpuusage_ps', 'pxdwngrdserial_pt', 'pxdwngrd50_ps', 'libcache_hit_pct', 'inmem_sort_pct', 'dumpAvail', 'dumpTotal', 'dumpUsedPercent', 'dumpUsed', 'scn_intrinsic_growth_rate', 'max_tot_cpu_usage_ps', 'avg_tot_cpu_usage_ps', 'id', '_version_']

In [158]:
"server" in a

False

In [155]:
prediction_df = makePredictionFrame(model, cubes_frames, last_timestamp)

['count', 'transactions_ps', 'opt_sga_size', 'sga_total', 'fixed_sga', 'java_pool', 'shared_pool', 'buffer_cache', 'pga_total', 'other_sga_memory', 'log_buffer', 'large_pool', 'streams_pool', 'total_pga_allocated', 'hardParseBindMismElTimePs', 'javaExecutionElapsedTimePs', 'plsqlEexecutionElapsedTimePs', 'connManageCallElapsedTimePs', 'failedParseElapsedTimePs', 'dbTimePs', 'hardParseSharCritElTimePs', 'plsqlCompilationElapsedTimePs', 'bgCpuTimePs', 'sqlExecuteElapsedTimePs', 'parseTimeElapsedPs', 'failedParseSharMemElTimePs', 'repeatedBindElapsedTimePs', 'dbCpuPs', 'rmanCpuTimeBackupRestorePs', 'inboundPlsqlRpcElapsedTimePs', 'sequenceLoadElapsedTimePs', 'hardParseElapsedTimePs', 'averageWaitTime', 'totalWaitsFG', 'averageWaitTimeFG', 'totalWaitTimeFG', 'totalWaits', 'totalWaitTime', 'shared_free_pct', 'large_free_pct', 'java_free_pct', 'session_usage', 'logons', 'user_limit', 'opencursors', 'process_usage', 'total_memory', 'genericErrors', 'archiveHungErrors', 'blockCorruptErrors', '

ValueError: could not convert string to float: 'baa0cea4-f366-4971-930f-2fecb34d7305'